# 19 · Concurrency

*Beyond the contest* · 6 exercises

Do several things at once safely: start work and wait for all of it, pass values between workers, cap the number of workers, protect shared state, and stop waiting when a result is late.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ch-19) · [Answer key](https://bloomcode.github.io/cp/python/answer-key.html#ch-19)

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.
import queue
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from concurrent.futures import ThreadPoolExecutor, TimeoutError

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    class _Raises:
        def __init__(self, kind):
            self.kind = kind

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            return issubclass(kind, self.kind)

    class pytest:  # noqa: N801
        raises = _Raises

### 19.1 · Start workers and wait for all  `Core`

**Task.** Write `parallel_squares(nums)` that starts one `threading.Thread` per number, each storing its number's square in its own slot of a results list, then `join`s every thread before returning the list.

**Example.** `[1, 2, 3]` -> `[1, 4, 9]`

**Big O.** O(N) work spread over N threads; each writes only its own slot, so no lock is needed.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-19-1) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-19-1)

In [ ]:
# 19.1 Start workers and wait for all
def parallel_squares(nums):
    raise NotImplementedError

In [ ]:
def test_ex_19_1():
    assert parallel_squares([1, 2, 3]) == [1, 4, 9] and parallel_squares([]) == [] and parallel_squares(list(range(50))) == [i * i for i in range(50)]


test_ex_19_1()
print('19.1 passed')

### 19.2 · A producer and a consumer  `Core`

**Task.** Write `pipeline_total(nums)` with two threads joined by a `queue.Queue`: a producer thread puts each number and then `None` to say it is finished, while a consumer thread takes items until it sees `None`, adding them up. Return the consumer's total after joining both.

**Example.** `[4, 5, 6]` -> `15`

**Big O.** O(N): every value passes through the queue once; the queue does the locking and the waiting.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-19-2) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-19-2)

In [ ]:
# 19.2 A producer and a consumer
def pipeline_total(nums):
    raise NotImplementedError

In [ ]:
def test_ex_19_2():
    assert pipeline_total([4, 5, 6]) == 15 and pipeline_total([]) == 0 and pipeline_total(list(range(10_000))) == 49_995_000


test_ex_19_2()
print('19.2 passed')

### 19.3 · A pool with a fixed number of workers  `Practice`

**Task.** Write `pool_map(fn, items, workers)` that applies `fn` to every item using at most `workers` threads at a time and returns the results in input order. Use a `ThreadPoolExecutor(max_workers=workers)` in a `with` block and its `map`.

**Example.** `lambda x: x * 10, [1, 2, 3], 2` -> `[10, 20, 30]`

**Big O.** About ceil(N / W) rounds of work for W workers; the results keep the input order however the work finishes.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-19-3) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-19-3)

In [ ]:
# 19.3 A pool with a fixed number of workers
def pool_map(fn, items, workers):
    raise NotImplementedError

In [ ]:
def test_ex_19_3():
    assert pool_map(lambda x: x * 10, [1, 2, 3], 2) == [10, 20, 30] and pool_map(str, [], 3) == []
    lock, active, peak = threading.Lock(), [0], [0]

    def slow(x):
        with lock:
            active[0] += 1
            peak[0] = max(peak[0], active[0])
        time.sleep(0.01)
        with lock:
            active[0] -= 1
        return -x

    assert pool_map(slow, list(range(12)), 3) == [-x for x in range(12)] and 1 <= peak[0] <= 3


test_ex_19_3()
print('19.3 passed')

### 19.4 · A counter protected by a lock  `Core`

**Task.** Write `locked_count(threads, increments)` that starts `threads` threads, each adding 1 to a shared counter `increments` times, and returns the final count. Guard every update with one `threading.Lock` (`with lock:`), because `count += 1` is a read, an add and a write that two threads can interleave.

**Example.** `4, 1000` -> `4000`

**Big O.** O(T * K) increments; the lock makes each read-modify-write happen as one step, so none is lost.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-19-4) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-19-4)

In [ ]:
# 19.4 A counter protected by a lock
def locked_count(threads, increments):
    raise NotImplementedError

In [ ]:
def test_ex_19_4():
    assert locked_count(4, 1000) == 4000 and locked_count(1, 0) == 0 and locked_count(8, 2500) == 20_000


test_ex_19_4()
print('19.4 passed')

### 19.5 · Stop waiting after a timeout  `Practice`

**Task.** Write `result_within(task, seconds)` that runs `task()` on a worker thread and returns `(True, result)` if it finishes within `seconds`, or `(False, None)` otherwise, without waiting for a slow task to end. Submit it to a `ThreadPoolExecutor`, call `future.result(timeout=seconds)`, and `shutdown(wait=False)`.

**Example.** `lambda: 42, 1.0` -> `(True, 42)`; a task that sleeps 0.3 s with `0.05` -> `(False, None)`

**Big O.** Waits at most the timeout, however long the task takes.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-19-5) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-19-5)

In [ ]:
# 19.5 Stop waiting after a timeout
def result_within(task, seconds):
    raise NotImplementedError

In [ ]:
def test_ex_19_5():
    assert result_within(lambda: 42, 1.0) == (True, 42)
    start = time.monotonic()
    assert result_within(lambda: time.sleep(0.3), 0.05) == (False, None) and time.monotonic() - start < 0.25


test_ex_19_5()
print('19.5 passed')

### 19.6 · Split a sum across workers  `Practice`

**Task.** Write `parallel_sum(nums, workers)` that cuts `nums` into `workers` contiguous chunks of near-equal size, sums each chunk on its own thread (a `ThreadPoolExecutor`), and adds the partial sums.

**Example.** `list(range(1, 101)), 4` -> `5050`

**Big O.** O(N / W) per worker for W workers, then O(W) to combine; each worker owns one slot of the partial results.

**Note.** In CPython, threads take turns running Python code (the GIL), so this shows the pattern rather than a speed-up; use processes for CPU-heavy work.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-19-6) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-19-6)

In [ ]:
# 19.6 Split a sum across workers
def parallel_sum(nums, workers):
    raise NotImplementedError

In [ ]:
def test_ex_19_6():
    assert parallel_sum(list(range(1, 101)), 4) == 5050 and parallel_sum([], 3) == 0 and parallel_sum([5], 8) == 5
    assert parallel_sum(list(range(1000)), 7) == 499_500


test_ex_19_6()
print('19.6 passed')